### **PHY 321 Semester Project** ###

**Names of collaborators:**
- Matrim Cirullo-Nesbitt
- Abhishek Vilekar
- Andrew Gabaldon


### **How Does Other Forces Influence a Golf Ball's Trajectory?** ###

#### **Introduction**

A golf ball acting as a moving projectile under gravity, while appearing to be a simple system, is in reality under the influence of multiple complex effects. Our project aims to determine the extent of how the trajectory a golf ball deviates due to its dimples and the velocity/spin-dependent Magnus effect. We aim to do this in three steps. First, we perform a boundary layer analysis of a specific golf ball (to reduce complexity) using a velocity inflow method. Then, using the results of such, we create a dataset to approximate and fit a function for the $S(v)$ parameter in the Magnus effect. Finally, we integrate the equations of motion of the macroscopic system to calculate and visualize trajectories. We compare the difference between an idealised model of a projectile and our model (gravity only and gravity + newtonian drag). The accuracy of our refined model will be tested using real-life data of golf-ball trajectories.

First, we shall import the necessary libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

#### **What is the Magnus Effect?**

The Magnus Effect is a physical phenomenon where a spinning object (golf ball) moving through a fluid medium (air) curves away from the expected path. This effect is very prominent in various sports like soccer, tennis, and in our project, golf. The Magnus effect is what causes the balls to curve in different directions depending on the spin. This is caused by the spin of the ball creating high and low pressure zones on each side of the ball, creating an induced force perpendicular to the path of motion. This is evident in the sport of golf when the ball is given topspin creating an induced force towards the ground sending the ball farther forward, or backspin creating an induced force towards the sky sending the ball higher in the air along its path.


\
\
<img src="magnus.png" width="500">



#### **Basic Projectile Under Gravity and Drag** ####

A simplified situation of a ball of under gravity can be compared to case with newtonian drag. The ball initially starts with an initial velocity $v_0$ at an angle $\theta$. Our coordinate system is positive upwards in y and positive to the right in x.

Other assumptions include a uniform density of air, a smooth ball and no spin of the ball. The diameter of the ball is taken to be 1.680 inches which is the minimum diameter allowed according to USGA Rules of Golf. Gravity only acts downwards thus there is an acceleration in the y-component of velocity. We can then write the equations of motion in x, y and the velocity components,

$$
\large
\frac{dx}{dt} = v_x
$$

$$
\large
\frac{dy}{dt} = v_y
$$

$$
\large
\frac{dv_x}{dt} = 0
$$

$$
\large
\frac{dv_y}{dt} = -g
$$

Accounting for newtonian drag, a extra term of force is added,

$$
\large
\vec{F}_D
=
-\frac{1}{2}\,\rho\,C_d\,A\,v\,\mathbf{v}
$$

$$
\large
\mathbf{v} = v_x\,\hat{x} + v_y\,\hat{y}, 
\quad
v = \|\vec{v}\| = \sqrt{v_x^2 + v_y^2}, 
\quad
$$
Where $\rho$ is the air density, A is the cross-sectional area and $C_d$ is the drag coefficient.
The drag force is a vector equation affecting both x and y components of velocity, modifying our equations of motion into,

$$
\large
\frac{dx}{dt} = v_x
$$

$$
\large
\frac{dy}{dt} = v_y
$$

$$
\large
\frac{dv_x}{dt}
=
-\frac{1}{2m}\,\rho\,C_d\,A\,v\,v_x
$$

$$
\large
\frac{dv_y}{dt}
=
-g \;-\; \frac{1}{2m}\,\rho\,C_d\,A\,v\,v_y
$$

$$
\large
v = \sqrt{v_x^2 + v_y^2}
$$

We can then define our constants and numerically solve the equations of motion,

In [ ]:
v0 = 160*5/18 #Assume initial velocity of 160km/h
theta0 = np.deg2rad(15) #Assume launch angle of 15 degrees
g = 9.81 #m/s^2
rho_0 = 1.293 #kg/m3
Cd = 0.47  #Drag coefficient relevant for the case of a projectile such as a golf ball
r = 0.042672/2 #Golf ball radius in m from a diameter of 1.68 inches
A = 4*np.pi*r**2
m = 0.04592623 #Mass of golf ball taken to be the maximum allowed by the USGA Rules of Golf (1.62 ounces) converted to kg

def newt(state, t):
    x, y, v_x, v_y = state
    return [v_x, v_y, 0, -g]

state0 = [0, 0, v0*np.cos(theta0), v0*np.sin(theta0)]
t = np.linspace(0, 2.5, 1000)
sol_newt = odeint(newt, state0, t)

def newt_drag(state, t):
    x, y, v_x, v_y = state
    return [v_x, v_y, -1/(2*m)*rho_0*Cd*A*np.sqrt(v_x**2+v_y**2)*v_x, -g-1/(2*m)*rho_0*Cd*A*np.sqrt(v_x**2+v_y**2)*v_y]

sol_drag = odeint(newt_drag, state0, t)

plt.figure(figsize=(7,6))
plt.title('Trajectory of a Golf Ball')
plt.plot(sol_newt[:,0], sol_newt[:,1], color='b', label='No Drag')
plt.plot(sol_drag[:,0], sol_drag[:,1], color='red', label='Newtonian Drag')
plt.ylim(0,7)
plt.xlabel('x [m]')
plt.ylabel('y [m]')
plt.legend()
plt.grid()
plt.show()

### Lots a stuff here.

#### **Obtaining s(v) Parameter** ####

In [ ]:
import pandas as pd

golf_sim = pd.read_csv("force_output.csv") #Read Pandas Dataframe
golf_sim.columns = golf_sim.columns.str.strip().str.replace("'", "") #Make sure columns are readable by code

golf_sim["Angular velocity (rad/s)"] = pd.to_numeric(golf_sim["Angular velocity (rad/s)"], errors="coerce") #Fix Datatype issue

golf_rot = golf_sim[golf_sim["Angular velocity (rad/s)"] > 0] #Only view rotating values (division by 0 breaks simulation)
golf_rot = golf_rot.reset_index(drop=True) # re-index after culling
v = golf_rot['Velocity (m/s)']
Fn = golf_rot['Force normal v (N)'] #Establish v, Fn, and omega values for function
omega = golf_rot['Angular velocity (rad/s)']

In [ ]:
# separate for generating averaged values
buffer = []
avg_vals = []
for i in range(1,len(v)):
    if v[i] == v[i-1]:
        buffer.append((v[i-1],Fn[i-1],omega[i-1]))
        #print(buffer) debugging
        try:
            if v[i] != v[i+1]:
                buffer.append((v[i],Fn[i],omega[i]))
                buffer = np.array(buffer)
                v_avg = np.average(buffer[:,0])
                #print(buffer) debugging
                Fn_avg = np.average(buffer[:,1])
                omega_avg = np.average(buffer[:,2])
                avg_vals.append((v_avg,Fn_avg,omega_avg))
                buffer = []
        except:
            buffer.append((v[i],Fn[i],omega[i]))
            buffer = np.array(buffer)
            v_avg = np.average(buffer[:,0])
            Fn_avg = np.average(buffer[:,1])
            omega_avg = np.average(buffer[:,2])
            avg_vals.append((v_avg,Fn_avg,omega_avg))
            buffer = []
avg_vals = np.array(avg_vals)


In [ ]:
from scipy.interpolate import interp1d

v = avg_vals[:,0] 
Fn = avg_vals[:,1]
omega = avg_vals[:,2]

s = 100 * np.abs( Fn / ( v * omega ) ) #calculate s for each point (our method treats it as a percentage, so multiply by 100)
s_of_v = interp1d(v, s, kind="cubic", fill_value="extrapolate") #Use interpolation function to solve for s(v) parameter

print(s_of_v(20)) #s(v) at 20 m/s
interp_v = np.linspace(0,70,1000)
plt.plot(interp_v,s_of_v(interp_v))
plt.show()

As can be seen, this is incredibly nonlinear, with extremely non-trivial behavior. This is sort of what we expect though, as recorded in other experimental works on the magnus effect, specifically for golf balls, the $S(v)$ parameter is very much in flux, between some values. The range of values that $S(v)$ takes is very close to the expected values of generally between 0.1 and 0.3, with a total range of 0.1 to 1.4.